In [28]:
import requests
from fredapi import Fred
import pandas as pd
import plotly.graph_objects as go
import yfinance as yf
import pandas as pd
import numpy as np
import requests
import zipfile
import io
import plotly.express as px

# FRED Data

In [2]:
fred = Fred(api_key='40b5fe20eb1362f04fa6a2f3c6a86fed')

In [3]:
FRED_CODES = {
    'DEXJPUS' : ["Japanese Yen to U.S. Dollar Spot Exchange Rate", 'D'],
    'IRLTLT01JPM156N' : ['JPY 10Y', 'M'],
    'IR3TIB01JPM156N' : ['JPY 3M', 'M'], 
    'IRSTCI01JPM156N' : ['JPY Interbank', 'M'],
    'JPINTDUSDJPY' : ['JPY YCC', 'D'],
    
}
# 'PPPTTLJPA618NUPN' : 'JPY PPP'
start_date = pd.to_datetime('2000-01-01').date()
end_date = pd.to_datetime('2026-01-01').date()

In [4]:
def get_fred_data(FRED_CODES,start_date , end_date):
    start_date = pd.to_datetime(start_date).date()
    end_date = pd.to_datetime(end_date).date()
    df_list = []
    ## get all series from FRED
    for code in FRED_CODES.keys():
        print(code,": ",FRED_CODES[code][0], "\n" )

        
        series = fred.get_series(code)
        series = series.to_frame(name=f'{FRED_CODES[code][0]}({FRED_CODES[code][1]})')

        series.index = pd.to_datetime(series.index)
        series = series.loc[start_date:end_date ]
        series.index = series.index.date
        
        df_list.append(series)

    ## Find highest Date and lowest date from all the series
    highest_start = start_date
    lowest_end = end_date
    freq_list = []
    for df in df_list:
        print(f"{df.columns.values.tolist()[0]}: ({df.index[0]} ->>, {df.index[-1]}), Freq: {df.columns.values.tolist()[0][-2]}")
        freq_list.append(df.columns.values.tolist()[0][-2])

        if df.index[0]>highest_start:
            highest_start = df.index[0]
            
        if df.index[-1]<lowest_end:
            lowest_end = df.index[-1]

    # Find lowest Frequency
    if "D" in freq_list:
        freq = 'D'
    elif "W" in freq_list:
        freq = 'W'
    elif "M" in freq_list:
        freq = 'M'
    elif "Q" in freq_list:
        freq = 'Q'
    elif "Y" in freq_list:   
        freq = 'Y'
    else:
        freq = 'D'


    date_range = pd.date_range(start = highest_start, end = lowest_end, freq = freq)
    date_range

    Merged = pd.DataFrame()

    for i in range(len(df_list)):
        df_list[i] = df_list[i].reindex(date_range, method = 'ffill')
        df_list[i] = df_list[i].bfill()

    all_df = pd.concat(df_list, axis=1)
    return all_df

In [5]:
jap = get_fred_data(FRED_CODES, start_date='2016-01-01', end_date='2025-12-31')

DEXJPUS :  Japanese Yen to U.S. Dollar Spot Exchange Rate 



/var/folders/hs/v5c0lrnj61z3tp745vdhh0m40000gn/T/ipykernel_12529/3978148300.py:14: Pandas4Warning: Slicing with a datetime.date object is deprecated. Explicitly cast to Timestamp instead.
  series = series.loc[start_date:end_date ]


IRLTLT01JPM156N :  JPY 10Y 



/var/folders/hs/v5c0lrnj61z3tp745vdhh0m40000gn/T/ipykernel_12529/3978148300.py:14: Pandas4Warning: Slicing with a datetime.date object is deprecated. Explicitly cast to Timestamp instead.
  series = series.loc[start_date:end_date ]


IR3TIB01JPM156N :  JPY 3M 



/var/folders/hs/v5c0lrnj61z3tp745vdhh0m40000gn/T/ipykernel_12529/3978148300.py:14: Pandas4Warning: Slicing with a datetime.date object is deprecated. Explicitly cast to Timestamp instead.
  series = series.loc[start_date:end_date ]


IRSTCI01JPM156N :  JPY Interbank 



/var/folders/hs/v5c0lrnj61z3tp745vdhh0m40000gn/T/ipykernel_12529/3978148300.py:14: Pandas4Warning: Slicing with a datetime.date object is deprecated. Explicitly cast to Timestamp instead.
  series = series.loc[start_date:end_date ]


JPINTDUSDJPY :  JPY YCC 

Japanese Yen to U.S. Dollar Spot Exchange Rate(D): (2016-01-01 ->>, 2025-12-31), Freq: D
JPY 10Y(M): (2016-01-01 ->>, 2025-12-01), Freq: M
JPY 3M(M): (2016-01-01 ->>, 2025-12-01), Freq: M
JPY Interbank(M): (2016-01-01 ->>, 2025-12-01), Freq: M
JPY YCC(D): (2016-01-01 ->>, 2025-12-31), Freq: D


/var/folders/hs/v5c0lrnj61z3tp745vdhh0m40000gn/T/ipykernel_12529/3978148300.py:14: Pandas4Warning: Slicing with a datetime.date object is deprecated. Explicitly cast to Timestamp instead.
  series = series.loc[start_date:end_date ]


In [6]:
fig = go.Figure()
fig.add_trace(go.Scatter(x = jap.index, y = jap['Japanese Yen to U.S. Dollar Spot Exchange Rate(D)']))

In [7]:
fig = go.Figure()
fig.add_trace(go.Scatter(x = jap.index, y = jap['JPY 10Y(M)'], name='JPY 10Y(M)'))
fig.add_trace(go.Scatter(x = jap.index, y = jap['JPY 3M(M)'], name='JPY 3M(M)'))
fig.add_trace(go.Scatter(x = jap.index, y = jap['JPY Interbank(M)'], name='JPY Interbank(M)'))

# COT Data

In [23]:
def download_cot_data(years, report_type='combined'):
    """
    Downloads raw CFTC Traders in Financial Futures (TFF) data.

    Args:
        years       : list of int — e.g. [2020, 2021, 2022]
        report_type : 'futures' or 'combined' (futures + options)

    Returns:
        pd.DataFrame — raw unmodified TFF data
    """
    prefix_map = {
        'futures':  'fut_fin_xls',
        'combined': 'com_fin_xls',
    }
    assert report_type in prefix_map, f"report_type must be one of {list(prefix_map.keys())}"
    prefix = prefix_map[report_type]

    headers = {
        'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64)',
        'Referer':    'https://www.cftc.gov/MarketReports/CommitmentsofTraders/index.htm',
    }

    dfs = []
    for year in years:
        url = f"https://www.cftc.gov/files/dea/history/{prefix}_{year}.zip"
        print(f"Fetching {report_type} COT {year} from {url}...")
        try:
            response = requests.get(url, headers=headers, timeout=30)
            response.raise_for_status()
            zf  = zipfile.ZipFile(io.BytesIO(response.content))
            xls = zf.open(zf.namelist()[0]).read()
            dfs.append(pd.read_excel(io.BytesIO(xls), engine='xlrd'))
            print(f"  ✓ {year} fetched successfully")
        except requests.HTTPError as e:
            print(f"  ✗ HTTP error for {year}: {e}")
        except zipfile.BadZipFile:
            print(f"  ✗ Bad zip file for {year}")
        except Exception as e:
            print(f"  ✗ Unexpected error for {year}: {e}")

    if not dfs:
        raise RuntimeError("No data fetched — check years and network connection")

    return pd.concat(dfs, ignore_index=True)

In [24]:
def find_dominant_contracts(df, instrument_name):
    """
    For a given instrument, shows which contract names exist,
    how many weeks each was dominant, and the date range of dominance.

    Use this BEFORE updating INSTRUMENT_MAP to understand what to include.

    Args:
        df              : raw COT DataFrame
        instrument_name : str — search term e.g. 'EURO FX', 'NASDAQ', 'BITCOIN'
        top_n           : int — how many contracts to show

    Returns:
        pd.DataFrame — contract names, dominance count, date ranges
    """
    # Find all contracts matching the search term and print the variant names
    df_filtered = df[df['Market_and_Exchange_Names'].str.contains(instrument_name)]
    df_filtered = df_filtered[['Market_and_Exchange_Names', 'Report_Date_as_MM_DD_YYYY', 'Open_Interest_All']]
    df_filtered_pivot = df_filtered.pivot_table(
    index='Report_Date_as_MM_DD_YYYY',
    columns='Market_and_Exchange_Names',
    values='Open_Interest_All',
    aggfunc='first' ) # in case of duplicates, take first


    variant_names = (df_filtered_pivot.columns.values.tolist())
    print(f"Variants: For {instrument_name}: ") 
    for i in variant_names:
        print(i)
    if len(variant_names) == 0:
        print('Probably Searched Wrong')

    df_filtered_pivot['Total'] = df_filtered_pivot.sum(axis=1, )

    df_filtered_pivot_proportions = df_filtered_pivot.div(df_filtered_pivot['Total'], axis=0) * 100
    df_filtered_pivot_proportions = df_filtered_pivot_proportions.drop(columns='Total')
    return px.line(df_filtered_pivot_proportions)

In [25]:
def clean_financial_cot_data(df, instrument_map):
    ''' Dealer — financial institutions/dealers (often on the other side of client trades)
        Asset_Mgr — institutional investors like pension funds, mutual funds
        Lev_Money — leveraged money (hedge funds, CTAs)
        Other_Rept — other reportable traders
        NonRept — non-reportable (small traders below reporting thresholds)
        Traders - it's counting number of traders, knowing 47 dealers are long tells you less than knowing their total position size
        Concentration Report - Shows what % of open interest the top 4 and top 8 traders control'''
    
    # take only the columns we need
    df = df[['Market_and_Exchange_Names', 
    'Report_Date_as_MM_DD_YYYY', 
    'Open_Interest_All', 
    'Dealer_Positions_Long_All', 'Dealer_Positions_Short_All',
    'Asset_Mgr_Positions_Long_All','Asset_Mgr_Positions_Short_All',
    'Lev_Money_Positions_Long_All', 'Lev_Money_Positions_Short_All'
    ]]


    df['Report_Date_as_MM_DD_YYYY'] = pd.to_datetime(df['Report_Date_as_MM_DD_YYYY'])
    df = df.sort_values('Report_Date_as_MM_DD_YYYY')

    df['Instrument'] = df['Market_and_Exchange_Names'].map(instrument_map)
    df = df[df['Instrument'].notna()].copy()

    cols_to_sum = df.columns.values.tolist()
    cols_to_sum.remove('Market_and_Exchange_Names')
    cols_to_sum.remove('Report_Date_as_MM_DD_YYYY')
    cols_to_sum.remove('Instrument')
    df = (
    df.groupby(['Instrument', 'Report_Date_as_MM_DD_YYYY'])[cols_to_sum]
    .sum()
    .reset_index()
    )


    return df

In [ ]:
def process_COT(df_cot, instrument, start_date, end_date):
    """
    Args:
        df_cot      : cleaned COT dataframe (output of clean_cot_data())
        instrument  : str — e.g. 'SP500'
        start_date  : str — e.g. '2020-01-01'
        end_date    : str — e.g. '2022-12-31'
    """

    df_clean = df_cot[
    (df_cot['Instrument'] == instrument) &
    (df_cot['Report_Date_as_MM_DD_YYYY'] >= start_date) &
    (df_cot['Report_Date_as_MM_DD_YYYY'] <= end_date)
    ].copy()

    df_clean["Dealer Net"] = df_clean['Dealer_Positions_Long_All'] - df_clean['Dealer_Positions_Short_All']
    df_clean["Asset Manager Net"] = df_clean['Asset_Mgr_Positions_Long_All'] - df_clean['Asset_Mgr_Positions_Short_All'] 
    df_clean["Levered Net"] = df_clean['Lev_Money_Positions_Long_All'] - df_clean['Lev_Money_Positions_Short_All']

    df_clean["Dealer Long Proportion"] = df_clean['Dealer_Positions_Long_All']/df_clean['Open_Interest_All']
    df_clean["Asset Manager Long Proportion"] = df_clean['Asset_Mgr_Positions_Long_All']/df_clean['Open_Interest_All']
    df_clean["Levered Long Proportion"] = df_clean['Lev_Money_Positions_Long_All']/df_clean['Open_Interest_All']

    df_clean["Dealer Short Proportion"] = df_clean['Dealer_Positions_Short_All']/df_clean['Open_Interest_All']
    df_clean["Asset Manager Short Proportion"] = df_clean['Asset_Mgr_Positions_Short_All']/df_clean['Open_Interest_All']
    df_clean["Levered Short Proportion"] = df_clean['Lev_Money_Positions_Short_All']/df_clean['Open_Interest_All']


    df_clean["Dealer Crowding"] = (df_clean['Dealer_Positions_Long_All']+df_clean['Dealer_Positions_Short_All'])/df_clean['Open_Interest_All']
    df_clean["Asset Manager Crowding"] = (df_clean['Asset_Mgr_Positions_Long_All']+df_clean['Asset_Mgr_Positions_Short_All'])/df_clean['Open_Interest_All']
    df_clean["Levered Manager Crowding"] = (df_clean['Lev_Money_Positions_Long_All']+df_clean['Lev_Money_Positions_Short_All'])/df_clean['Open_Interest_All']

    
    ticker = TICKER_MAP[instrument]
    spy = yf.download(tickers=ticker, start=start_date, end=end_date)['Close']

    spy_tuesday = spy[spy.index.dayofweek == 1]
    spy_tuesday.index.name = 'Report_Date_as_MM_DD_YYYY'
    spy_tuesday.name = 'Price_Close'  # rename the Series before merging

    df_clean_merged = df_clean.merge(spy_tuesday, on='Report_Date_as_MM_DD_YYYY', how='inner')
    df_clean_merged = df_clean_merged.rename(columns={ticker: 'Price_Close'})
    df_clean_merged

    return df_clean_merged

In [52]:
cot_raw = download_cot_data([2017, 2018, 2019,2020, 2021, 2022, 2023, 2024, 2025, 2026])
cot_raw

Fetching combined COT 2017 from https://www.cftc.gov/files/dea/history/com_fin_xls_2017.zip...
  ✓ 2017 fetched successfully
Fetching combined COT 2018 from https://www.cftc.gov/files/dea/history/com_fin_xls_2018.zip...
  ✓ 2018 fetched successfully
Fetching combined COT 2019 from https://www.cftc.gov/files/dea/history/com_fin_xls_2019.zip...
  ✓ 2019 fetched successfully
Fetching combined COT 2020 from https://www.cftc.gov/files/dea/history/com_fin_xls_2020.zip...
  ✓ 2020 fetched successfully
Fetching combined COT 2021 from https://www.cftc.gov/files/dea/history/com_fin_xls_2021.zip...
  ✓ 2021 fetched successfully
Fetching combined COT 2022 from https://www.cftc.gov/files/dea/history/com_fin_xls_2022.zip...
  ✓ 2022 fetched successfully
Fetching combined COT 2023 from https://www.cftc.gov/files/dea/history/com_fin_xls_2023.zip...
  ✓ 2023 fetched successfully
Fetching combined COT 2024 from https://www.cftc.gov/files/dea/history/com_fin_xls_2024.zip...
  ✓ 2024 fetched successfully


,Market_and_Exchange_Names,As_of_Date_In_Form_YYMMDD,Report_Date_as_MM_DD_YYYY,CFTC_Contract_Market_Code,CFTC_Market_Code,CFTC_Region_Code,CFTC_Commodity_Code,Open_Interest_All,Dealer_Positions_Long_All,Dealer_Positions_Short_All,...,Conc_Gross_LE_4_TDR_Short_All,Conc_Gross_LE_8_TDR_Long_All,Conc_Gross_LE_8_TDR_Short_All,Conc_Net_LE_4_TDR_Long_All,Conc_Net_LE_4_TDR_Short_All,Conc_Net_LE_8_TDR_Long_All,Conc_Net_LE_8_TDR_Short_All,Contract_Units,CFTC_SubGroup_Code,FutOnly_or_Combined
0,CANADIAN DOLLAR - CHICAGO MERCANTILE EXCHANGE,171226,2017-12-26,090741,CME,0,90,134238,4812,55036,...,41.9,33.7,51.5,20.4,41.3,30.7,50.4,"(CONTRACTS OF CAD 100,000)",F10,Combined
1,CANADIAN DOLLAR - CHICAGO MERCANTILE EXCHANGE,171219,2017-12-19,090741,CME,0,90,172676,15819,81942,...,46.3,35.2,56.2,20.0,42.0,32.3,51.4,"(CONTRACTS OF CAD 100,000)",F10,Combined
2,CANADIAN DOLLAR - CHICAGO MERCANTILE EXCHANGE,171212,2017-12-12,090741,CME,0,90,165499,16251,79621,...,45.0,32.6,55.8,19.1,43.9,30.7,53.4,"(CONTRACTS OF CAD 100,000)",F10,Combined
3,CANADIAN DOLLAR - CHICAGO MERCANTILE EXCHANGE,171205,2017-12-05,090741,CME,0,90,171861,12245,82228,...,44.3,31.7,55.9,18.3,43.9,29.5,53.9,"(CONTRACTS OF CAD 100,000)",F10,Combined
4,CANADIAN DOLLAR - CHICAGO MERCANTILE EXCHANGE,171128,2017-11-28,090741,CME,0,90,170364,7049,84530,...,45.1,31.7,56.5,20.9,44.9,28.7,54.2,"(CONTRACTS OF CAD 100,000)",F10,Combined
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
25356,BBG COMMODITY - CHICAGO BOARD OF TRADE,260203,2026-02-03,221602,CBT,0,221,183456,126257,166798,...,88.7,89.2,96.6,82.0,88.6,87.3,95.0,($100 X INDEX),F90,Combined
25357,BBG COMMODITY - CHICAGO BOARD OF TRADE,260127,2026-01-27,221602,CBT,0,221,188271,125275,172650,...,89.4,89.8,96.8,82.8,89.4,87.9,95.4,($100 X INDEX),F90,Combined
25358,BBG COMMODITY - CHICAGO BOARD OF TRADE,260120,2026-01-20,221602,CBT,0,221,189350,125416,171958,...,88.5,89.7,96.0,82.6,88.5,87.7,94.8,($100 X INDEX),F90,Combined
25359,BBG COMMODITY - CHICAGO BOARD OF TRADE,260113,2026-01-13,221602,CBT,0,221,189092,125364,171894,...,88.6,89.7,96.2,82.7,88.6,87.8,95.0,($100 X INDEX),F90,Combined


In [74]:
cot_raw["Market_and_Exchange_Names"].unique()

<StringArray>
[            'CANADIAN DOLLAR - CHICAGO MERCANTILE EXCHANGE',
                 'SWISS FRANC - CHICAGO MERCANTILE EXCHANGE',
      'BRITISH POUND STERLING - CHICAGO MERCANTILE EXCHANGE',
                'JAPANESE YEN - CHICAGO MERCANTILE EXCHANGE',
                     'EURO FX - CHICAGO MERCANTILE EXCHANGE',
           'AUSTRALIAN DOLLAR - CHICAGO MERCANTILE EXCHANGE',
 'EURO FX/BRITISH POUND XRATE - CHICAGO MERCANTILE EXCHANGE',
  'EURO FX/JAPANESE YEN XRATE - CHICAGO MERCANTILE EXCHANGE',
               'RUSSIAN RUBLE - CHICAGO MERCANTILE EXCHANGE',
                'MEXICAN PESO - CHICAGO MERCANTILE EXCHANGE',
 ...
                                   'NANO XRP - LMX LABS LLC',
                         'SOL - CHICAGO MERCANTILE EXCHANGE',
                                'NANO SOLANA - LMX LABS LLC',
       'BITCOIN CASH PERP STYLE - COINBASE DERIVATIVES, LLC',
           'LITECOIN PERP STYLE - COINBASE DERIVATIVES, LLC',
           'DOGECOIN PERP STYLE - COINBASE DERIVATI

In [ ]:
cot_raw

In [75]:
find_dominant_contracts(cot_raw, 'BITCOIN')

Variants: For BITCOIN: 
BITCOIN - CHICAGO MERCANTILE EXCHANGE
BITCOIN CASH PERP STYLE - COINBASE DERIVATIVES, LLC
BITCOIN-USD - CBOE FUTURES EXCHANGE
MICRO BITCOIN - CHICAGO MERCANTILE EXCHANGE
NANO BITCOIN PERP STYLE - COINBASE DERIVATIVES, LLC


In [76]:
INSTRUMENT_MAP = {
    'BITCOIN - CHICAGO MERCANTILE EXCHANGE': 'BTC',
    'BITCOIN CASH PERP STYLE - COINBASE DERIVATIVES, LLC':'BTC',
    'BITCOIN-USD - CBOE FUTURES EXCHANGE': 'BTC',
    'MICRO BITCOIN - CHICAGO MERCANTILE EXCHANGE':'BTC',
    'NANO BITCOIN PERP STYLE - COINBASE DERIVATIVES, LLC':'BTC'
}

In [77]:
TICKER_MAP = {
    'BTC':   'BTC-USD',
}

In [78]:
cot_clean = clean_financial_cot_data(cot_raw, INSTRUMENT_MAP)

In [79]:
processed_cot_1 = process_COT(cot_clean, 'BTC', '2017-01-01', '2027-01-01')
processed_cot_1

[*********************100%***********************]  1 of 1 completed


,Instrument,Report_Date_as_MM_DD_YYYY,Open_Interest_All,Dealer_Positions_Long_All,Dealer_Positions_Short_All,Asset_Mgr_Positions_Long_All,Asset_Mgr_Positions_Short_All,Lev_Money_Positions_Long_All,Lev_Money_Positions_Short_All,Dealer Net,...,Dealer Long Proportion,Asset Manager Long Proportion,Levered Long Proportion,Dealer Short Proportion,Asset Manager Short Proportion,Levered Short Proportion,Dealer Crowding,Asset Manager Crowding,Levered Manager Crowding,Price_Close
0,BTC,2017-12-19,2821,106,0,0,0,255,353,106,...,0.037575,0.000000,0.090393,0.000000,0.000000,0.125133,0.037575,0.000000,0.215526,17776.699219
1,BTC,2017-12-26,3737,39,0,0,0,259,1138,39,...,0.010436,0.000000,0.069307,0.000000,0.000000,0.304522,0.010436,0.000000,0.373829,16099.799805
2,BTC,2018-01-02,4065,39,0,0,0,274,380,39,...,0.009594,0.000000,0.067405,0.000000,0.000000,0.093481,0.009594,0.000000,0.160886,14982.099609
3,BTC,2018-01-09,4624,39,0,0,0,521,654,39,...,0.008434,0.000000,0.112673,0.000000,0.000000,0.141436,0.008434,0.000000,0.254109,14595.400391
4,BTC,2018-01-16,4736,40,0,0,0,379,608,40,...,0.008446,0.000000,0.080025,0.000000,0.000000,0.128378,0.008446,0.000000,0.208404,11490.500000
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
420,BTC,2026-02-10,198946,11551,4032,7769,2084,45587,87960,7519,...,0.058061,0.039051,0.229143,0.020267,0.010475,0.442130,0.078328,0.049526,0.671273,68793.960938
421,BTC,2026-02-17,188651,10655,4122,7238,2075,61630,46436,6533,...,0.056480,0.038367,0.326688,0.021850,0.010999,0.246148,0.078330,0.049366,0.572836,67494.218750
422,BTC,2026-02-24,221025,13937,6226,7032,2159,69476,88414,7711,...,0.063056,0.031815,0.314335,0.028169,0.009768,0.400018,0.091225,0.041584,0.714354,64080.042969
423,BTC,2026-03-03,241369,9186,4300,7455,2557,108938,90425,4886,...,0.038058,0.030886,0.451334,0.017815,0.010594,0.374634,0.055873,0.041480,0.825968,68293.648438


In [80]:
processed_cot_1 = processed_cot_1.set_index('Report_Date_as_MM_DD_YYYY')

In [81]:
processed_cot_1

,Instrument,Open_Interest_All,Dealer_Positions_Long_All,Dealer_Positions_Short_All,Asset_Mgr_Positions_Long_All,Asset_Mgr_Positions_Short_All,Lev_Money_Positions_Long_All,Lev_Money_Positions_Short_All,Dealer Net,Asset Manager Net,...,Dealer Long Proportion,Asset Manager Long Proportion,Levered Long Proportion,Dealer Short Proportion,Asset Manager Short Proportion,Levered Short Proportion,Dealer Crowding,Asset Manager Crowding,Levered Manager Crowding,Price_Close
Report_Date_as_MM_DD_YYYY,,,,,,,,,,,,,,,,,,,,,
2017-12-19,BTC,2821,106,0,0,0,255,353,106,0,...,0.037575,0.000000,0.090393,0.000000,0.000000,0.125133,0.037575,0.000000,0.215526,17776.699219
2017-12-26,BTC,3737,39,0,0,0,259,1138,39,0,...,0.010436,0.000000,0.069307,0.000000,0.000000,0.304522,0.010436,0.000000,0.373829,16099.799805
2018-01-02,BTC,4065,39,0,0,0,274,380,39,0,...,0.009594,0.000000,0.067405,0.000000,0.000000,0.093481,0.009594,0.000000,0.160886,14982.099609
2018-01-09,BTC,4624,39,0,0,0,521,654,39,0,...,0.008434,0.000000,0.112673,0.000000,0.000000,0.141436,0.008434,0.000000,0.254109,14595.400391
2018-01-16,BTC,4736,40,0,0,0,379,608,40,0,...,0.008446,0.000000,0.080025,0.000000,0.000000,0.128378,0.008446,0.000000,0.208404,11490.500000
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2026-02-10,BTC,198946,11551,4032,7769,2084,45587,87960,7519,5685,...,0.058061,0.039051,0.229143,0.020267,0.010475,0.442130,0.078328,0.049526,0.671273,68793.960938
2026-02-17,BTC,188651,10655,4122,7238,2075,61630,46436,6533,5163,...,0.056480,0.038367,0.326688,0.021850,0.010999,0.246148,0.078330,0.049366,0.572836,67494.218750
2026-02-24,BTC,221025,13937,6226,7032,2159,69476,88414,7711,4873,...,0.063056,0.031815,0.314335,0.028169,0.009768,0.400018,0.091225,0.041584,0.714354,64080.042969


In [82]:
processed_cot_1.columns.values.tolist()

['Instrument',
 'Open_Interest_All',
 'Dealer_Positions_Long_All',
 'Dealer_Positions_Short_All',
 'Asset_Mgr_Positions_Long_All',
 'Asset_Mgr_Positions_Short_All',
 'Lev_Money_Positions_Long_All',
 'Lev_Money_Positions_Short_All',
 'Dealer Net',
 'Asset Manager Net',
 'Levered Net',
 'Dealer Long Proportion',
 'Asset Manager Long Proportion',
 'Levered Long Proportion',
 'Dealer Short Proportion',
 'Asset Manager Short Proportion',
 'Levered Short Proportion',
 'Dealer Crowding',
 'Asset Manager Crowding',
 'Levered Manager Crowding',
 'Price_Close']

In [83]:
from plotly.subplots import make_subplots
fig = make_subplots(specs=[[{"secondary_y": True}]])
fig.add_trace(go.Scatter(x = processed_cot_1.index, y = processed_cot_1['Open_Interest_All'], name= 'Open_Interest_All'))
fig.add_trace(go.Scatter(x = processed_cot_1.index, y = processed_cot_1['Price_Close'], name= 'Price_Close'), secondary_y= True)
fig

In [84]:
from plotly.subplots import make_subplots
fig = make_subplots(specs=[[{"secondary_y": True}]])
fig.add_trace(go.Scatter(x = processed_cot_1.index, y = processed_cot_1['Dealer Net'], name= 'Dealer Net'))
fig.add_trace(go.Scatter(x = processed_cot_1.index, y = processed_cot_1['Asset Manager Net'], name= 'Asset Manager Net'))
fig.add_trace(go.Scatter(x = processed_cot_1.index, y = processed_cot_1['Levered Net'], name= 'Levered Net'))
fig.add_trace(go.Scatter(x = processed_cot_1.index, y = processed_cot_1['Price_Close'], name= 'Price_Close'), secondary_y= True)
fig